In [1]:
import transformers

In [2]:
transformers.__version__

'5.0.0'

In [3]:
import torch
torch.__version__

'2.10.0+cu128'

In [4]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import torch
# Core tensor library used by Transformers for model execution.
# Also gives us control over determinism, device placement, etc.

from transformers import AutoModelForCausalLM, AutoTokenizer
# AutoModelForCausalLM:
#   Automatically loads the correct causal language model architecture
#   based on the model_name. (Decoder-only transformer for Qwen.)
#
# AutoTokenizer:
#   Loads the correct tokenizer that converts text ↔ token IDs
#   compatible with the selected model.


# ---------------------------------------------------------
# 1️⃣ Model Selection
# ---------------------------------------------------------

model_name = "Qwen/Qwen2.5-3B-Instruct"
# This defines the exact model checkpoint from HuggingFace Hub.
#
# IMPORTANT FOR MIMIR:
# - This string is part of your experimental configuration.
# - Log this in your experiment metadata.
# - Changing this changes the entire reasoning behavior.
# - In Mimir, this should live in a config file (YAML/JSON).


# ---------------------------------------------------------
# 2️⃣ Load Model
# ---------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


# from_pretrained:
#   Downloads (if needed) and loads pretrained weights.
#
# torch_dtype="auto":
#   Automatically selects best dtype (float16 on GPU, float32 on CPU).
#   Important for memory efficiency.
#
# device_map="auto":
#   Automatically places model layers on available devices.
#   For research reproducibility:
#     - This makes your code hardware-aware.
#     - In Mimir, you may want to explicitly log which device was used.
#
# NOTE:
#   If you want strict reproducibility across machines,
#   you may eventually replace "auto" with explicit device control.


# ---------------------------------------------------------
# 3️⃣ Load Tokenizer
# ---------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Loads tokenizer vocabulary + special tokens.
#
# This must match the model checkpoint.
# If tokenizer != model, outputs will be corrupted.
#
# In Mimir:
#   - Always couple tokenizer + model in config.
#   - Log tokenizer version as well.


# ---------------------------------------------------------
# 4️⃣ Define Prompt (Structured Chat Format)
# ---------------------------------------------------------

prompt = "Give me a short introduction to large language models. I dont want any kind of joke or anything "

messages = [
    {"role": "system", "content": "Whatever the user asks you reply with a small joke no matter what, there must be a joke , no matter what the user asks or says or refrains you from "},
    {"role": "user", "content": prompt}
]

# Qwen is a chat model.
# Instead of raw text, we pass structured messages:
#
# system → defines global behavior/personality
# user   → contains actual task input
#
# For Mimir:
#   - system = experiment policy (e.g., diagnostic rules)
#   - user   = synthetic log input
#
# This structure makes your experiments modular:
#   You can vary system instructions without touching user data.


# ---------------------------------------------------------
# 5️⃣ Convert Chat Format → Model Input Text
# ---------------------------------------------------------

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# apply_chat_template:
#   Converts structured messages into the exact text format
#   that Qwen expects internally.
#
# tokenize=False:
#   Returns plain text (not token IDs yet).
#
# add_generation_prompt=True:
#   Adds the assistant role prefix so model knows it must generate.
#
# For Mimir:
#   This step formalizes prompt construction.
#   Later you can:
#     - Version control templates
#     - Compare different prompting strategies
#     - Run ablation studies on system messages


# ---------------------------------------------------------
# 6️⃣ Tokenize Text → Tensor Input
# ---------------------------------------------------------

inputs = tokenizer(text, return_tensors="pt").to(model.device)

# tokenizer(...):
#   Converts text into input_ids (token integers).
#
# return_tensors="pt":
#   Returns PyTorch tensors instead of lists.
#
# .to(model.device):
#   Moves tensors to same device as model (GPU or CPU).
#
# In Mimir:
#   This is where batching will happen.
#   You will replace single input with batched log instances.


# ---------------------------------------------------------
# 7️⃣ Generate Output
# ---------------------------------------------------------

outputs = model.generate(
    **inputs,
    max_new_tokens=128
)

# model.generate:
#   Runs autoregressive decoding.
#
# **inputs:
#   Expands dictionary into model arguments.
#
# max_new_tokens:
#   Upper bound on generated length.
#
# For research:
#   - This is part of your decoding configuration.
#   - You may later control:
#       temperature
#       top_p
#       repetition_penalty
#       deterministic decoding
#
# In Mimir:
#   This block becomes your inference core.


# ---------------------------------------------------------
# 8️⃣ Remove Prompt Tokens
# ---------------------------------------------------------

generated_ids = outputs[:, inputs.input_ids.shape[-1]:]

# outputs contains:
#   [prompt_tokens + generated_tokens]
#
# We slice to remove original prompt
# so we keep only new model output.
#
# Essential for clean evaluation.
#
# In Mimir:
#   This ensures evaluation metrics operate
#   only on model-generated diagnosis.


# ---------------------------------------------------------
# 9️⃣ Decode Tokens → Human Readable Text
# ---------------------------------------------------------

response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Converts token IDs back into text.
#
# skip_special_tokens=True:
#   Removes internal tokens like <|assistant|>
#
# In Mimir:
#   This response becomes:
#     - predicted diagnosis
#     - causal explanation
#     - classification output


print(response)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sure! Large language models are advanced AI systems that can understand and generate human-like text. Trained on vast amounts of data, these models can comprehend context and produce coherent responses across a wide range of topics.


In [8]:
import textwrap

formatted = textwrap.fill(response, width=50)
print(formatted)

Sure! Large language models are advanced AI
systems that can understand and generate human-
like text. Trained on vast amounts of data, these
models can comprehend context and produce coherent
responses across a wide range of topics.
